# 🧠 BERTopic — Retweet Topic Modeling

End-to-end pipeline for discovering topics and their retweet engagement stats.

**Pipeline overview:**
1. Load & inspect data
2. Clean text
3. Define embedding model
4. Compute & cache embeddings
5. Define representation models
6. Define UMAP, HDBSCAN, Vectorizer
7. Train BERTopic
8. Compute mean & std of retweets per cluster
9. Visualise results
10. Save model

## 📂 1. Load Data

In [23]:
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

DB_PATH = "../data/tweetsCleanedDB.sqlite"
TABLE_NAME = "tweets"

conn = sqlite3.connect(DB_PATH)

print(f"Processing table: {TABLE_NAME}")

cols = pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["name"].tolist()

df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
print(f"Loaded {len(df)} rows")

if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['date'] = df['Timestamp'].dt.date
    df['time'] = df['Timestamp'].dt.time
    df['day_of_week'] = df['Timestamp'].dt.day_name()
    df['hour'] = df['Timestamp'].dt.hour
    df['year'] = df['Timestamp'].dt.year

print(f"\n{'='*50}")
print(f"Columns: {df.columns.tolist()}")
summary = pd.DataFrame([(TABLE_NAME, len(df))], columns=["table_name", "num_rows"])
print(tabulate(summary, headers="keys", tablefmt="github", showindex=False))

# Column datatypes
print(f"\n{'='*50}")
print("Column Datatypes:")
dtype_df = pd.DataFrame({
    "column": df.dtypes.index,
    "pandas_dtype": df.dtypes.values.astype(str),
    "sqlite_dtype": pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["type"].tolist() + ["derived"] * (len(df.columns) - len(cols))
})
print(tabulate(dtype_df, headers="keys", tablefmt="github", showindex=False))

Processing table: tweets
Loaded 9909 rows

Columns: ['tweet_id', 'username', 'text', 'retweets', 'likes', 'timestamp', 'date', 'time', 'hour', 'day_name', 'year_week', 'year_month', 'year']
| table_name   |   num_rows |
|--------------|------------|
| tweets       |       9909 |

Column Datatypes:
| column     | pandas_dtype   | sqlite_dtype   |
|------------|----------------|----------------|
| tweet_id   | object         | TEXT           |
| username   | object         | TEXT           |
| text       | object         | TEXT           |
| retweets   | int64          | INTEGER        |
| likes      | int64          | INTEGER        |
| timestamp  | object         | TIMESTAMP      |
| date       | object         | DATE           |
| time       | object         | TEXT           |
| hour       | int64          | INTEGER        |
| day_name   | object         | TEXT           |
| year_week  | object         | TEXT           |
| year_month | object         | TEXT           |
| year       | 

## 📦 2. Imports & Setup

In [24]:
import os
import re
import numpy as np
import pandas as pd
import torch
import openai

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI

print('✅ All imports successful.')

## 🧹 3. Text Cleaning

In [25]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)           # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)                 # strip # but keep word
    text = re.sub(r'^rt\s+', '', text, flags=re.MULTILINE)# remove RT prefix
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

docs = df['text_clean'].tolist()
print(f'✅ Cleaned {len(docs)} documents.')
print(f'Sample: {docs[0]}')


✅ Cleaned 9909 documents.
Sample: party least receive say or single prevent prevent husband affect may himself cup style evening protect effect another themselves stage perform possible try tax share style television with successful much sell development economy effect


## 🔡 4. Embedding Model

In [26]:
# ⚡ SPEED: Using smaller L6 model (2x faster than L12, same dim)
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# ⚡ SPEED: Cap training corpus to 50k docs for sub-5-min runs
MAX_DOCS = 50_000

# Auto-detect best available device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'▶ Using device: {device}')

embedding_model = SentenceTransformer(EMBED_MODEL, device=device)

if device == 'cuda':
    embedding_model = embedding_model.half()  # FP16: ~2x throughput, half VRAM


## 🔢 5. Compute & Cache Embeddings

In [27]:
# ⚡ SPEED: Subsample to MAX_DOCS and cache embeddings
import random
random.seed(42)

all_docs = list(docs)
if len(all_docs) > MAX_DOCS:
    print(f'Subsampling {MAX_DOCS:,} / {len(all_docs):,} docs for speed...')
    indices = random.sample(range(len(all_docs)), MAX_DOCS)
    indices.sort()
    docs_train = [all_docs[i] for i in indices]
    # Also keep a sub-df for retweet stats
    df_train = df.iloc[indices].reset_index(drop=True)
else:
    docs_train = all_docs
    df_train = df.copy()

CACHE_PATH = 'embeddings_cache_l6_fast.npy'

if os.path.exists(CACHE_PATH):
    print(f"Loading cached embeddings from '{CACHE_PATH}'...")
    embeddings_train = np.load(CACHE_PATH)
    print(f'✅ Loaded from cache — shape: {embeddings_train.shape}')
else:
    print(f'Computing embeddings for {len(docs_train):,} docs...')
    batch_size = 512 if device == 'cuda' else (128 if device == 'mps' else 64)

    embeddings_train = embedding_model.encode(
        docs_train,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    )

    np.save(CACHE_PATH, embeddings_train)
    print(f"✅ Done — shape: {embeddings_train.shape}. Cached to '{CACHE_PATH}'")

# Keep backward compat
embeddings = embeddings_train


## 🎭 6. Representation Models

In [28]:
# ⚡ SPEED: Only fast local models — skip OpenAI (adds API latency)
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

keybert_model = KeyBERTInspired(top_n_words=5)
mmr_model     = MaximalMarginalRelevance(diversity=0.3, top_n_words=5)

representation_model = {
    'KeyBERT': keybert_model,
    'MMR':     mmr_model,
}

print(f'Representation models: {list(representation_model.keys())}')
print('ℹ️  OpenAI labelling skipped for speed — re-enable after training if needed.')


In [ ]:
# ⚡ SPEED + ROBUSTNESS: explicit safe vectorizer settings
vectorizer_model = CountVectorizer(
    max_features=5_000,
    min_df=1,
    max_df=1.0,
    ngram_range=(1, 2),
    strip_accents='unicode',
)

ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True,
)

print('✅ Pipeline components defined.')
print('Vectorizer config:', vectorizer_model.get_params())


In [ ]:
# ── Fit BERTopic on training documents ─────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

assert len(docs_train) > 0, 'docs_train is empty'
assert len(docs_train) == len(embeddings_train), (
    f'mismatch: {len(docs_train)} docs vs {len(embeddings_train)} embeddings'
)

vectorizer_model = CountVectorizer(
    max_features=5_000,
    min_df=2,              # ⚡ min_df=2 prunes rare unigrams faster
    max_df=0.95,
    ngram_range=(1, 2),
    strip_accents='unicode',
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    verbose=True,
    calculate_probabilities=False,
)

import time
t0 = time.time()
print(f'Fitting BERTopic on {len(docs_train):,} documents...')
topics, probs = topic_model.fit_transform(docs_train, embeddings=embeddings_train)
elapsed = time.time() - t0
print(f'⏱  fit_transform took {elapsed/60:.1f} min ({elapsed:.0f} s)')

df_train['topic_id'] = topics
# Keep df in sync for downstream cells
df = df_train

print(f'✅ BERTopic fitted on {len(docs_train):,} documents.')
print(f'   Topics found: {topic_model.get_topic_info().shape[0] - 1} (excluding outlier cluster -1)')
topic_model.get_topic_info().head(10)


In [30]:
# ── Initialise BERTopic ───────────────────────────────────────────────────────
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    verbose=True,
    calculate_probabilities=False,
)

print('✅ BERTopic model initialised.')


✅ BERTopic model initialised.


## 🚀 7. Fit BERTopic

In [ ]:
# ── Fit BERTopic on all documents ────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

assert len(docs) > 0, 'docs is empty after cleaning'
assert len(docs) == len(embeddings), f'mismatch: {len(docs)} docs vs {len(embeddings)} embeddings'

# Rebuild these objects here so the cell works even if the notebook kernel still has old state
vectorizer_model = CountVectorizer(
    max_features=5_000,
    min_df=1,
    max_df=1.0,
    ngram_range=(1, 2),
    strip_accents='unicode',
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    verbose=True,
    calculate_probabilities=False,
)

print('Fitting with vectorizer params:', topic_model.vectorizer_model.get_params())
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

df['topic_id'] = topics

print(f'✅ BERTopic fitted on {len(docs)} documents.')
print(f'   Topics found: {topic_model.get_topic_info().shape[0] - 1} (excluding outlier cluster -1)')
topic_model.get_topic_info().head(10)


## 📊 8. Retweet Stats per Cluster (Mean & Std)

In [ ]:
# ── Compute mean and std of retweets within every cluster ────────────────────
retweet_stats = (
    df[df['topic_id'] != -1]          # exclude outlier cluster
    .groupby('topic_id')['retweets']
    .agg(
        mean_retweets='mean',
        std_retweets='std',
        doc_count='count',
    )
    .reset_index()
    .sort_values('mean_retweets', ascending=False)
    .reset_index(drop=True)
)

# Merge with topic info for topic labels / top words
topic_info = topic_model.get_topic_info()
retweet_stats = retweet_stats.merge(
    topic_info[['Topic', 'Name']].rename(columns={'Topic': 'topic_id'}),
    on='topic_id',
    how='left',
)

print('✅ Retweet stats per cluster:')
print(retweet_stats.to_string(index=False))


## 📈 9. Visualisations

In [38]:
# ── Topic word scores ─────────────────────────────────────────────────────────
topic_model.visualize_barchart(top_n_topics=10, n_words=8)

In [39]:
# ── Topic similarity heatmap ──────────────────────────────────────────────────
topic_model.visualize_heatmap(top_n_topics=20)

In [40]:
# ── Intertopic distance map ───────────────────────────────────────────────────
topic_model.visualize_topics()

In [41]:
# ── Document-level topic map (sample 5000 docs to keep it fast) ───────────────
topic_model.visualize_documents(
    docs_train,
    embeddings=embeddings_train,
    sample=min(5000, len(docs_train)) / len(docs_train),
    hide_annotations=True,
)


## 💾 10. Save Model

In [ ]:
SAVE_PATH = '../data/bertopic_model'

# Attach retweet stats to model before saving
topic_model.retweet_stats_ = retweet_stats

topic_model.save(
    SAVE_PATH,
    serialization='pickle',
    save_ctfidf=True,
    save_embedding_model=EMBED_MODEL,
)
print(f"✅ Model saved to '{SAVE_PATH}'")

# To reload:
# loaded_model = BERTopic.load(SAVE_PATH)
# retweet_stats_loaded = loaded_model.retweet_stats_
